Prepare raw ITSM incident data for machine learning by removing leakage-prone fields, cleaning structure, and ensuring the dataset reflects only information available at prediction time.

In [1]:
#library imports
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
#Load data
df = pd.read_csv("../data/raw/incident_ml_raw.csv")
df.shape

(24918, 35)

From Notebook 01, we decided to drop:

Column	                          Reason
number	                          Identifier
sys_updated_at                 	  Data leakage risk
closed_at	                      Post-resolution
final_resolved_at	              Post-resolution
reassignment_count	              Zero variance
reopen_count	                  Zero variance

In [3]:
#drop columns

drop_cols = [
    'number',
    'sys_updated_at',
    'closed_at',
    'final_resolved_at',
    'reassignment_count',
    'reopen_count'
]

df = df.drop(columns=drop_cols, errors='ignore')
df.shape


(24918, 29)

Convert Time Columns

In [4]:
df['opened_at'] = pd.to_datetime(df['opened_at'])

C:\Users\rahul\AppData\Local\Temp\ipykernel_21512\1595204875.py:1: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['opened_at'] = pd.to_datetime(df['opened_at'])


In [5]:
#incident_state Reflects lifecycle progress
#is_active reflects Tells if incident is closed

leakage_cols = ['incident_state', 'is_active']
df = df.drop(columns=leakage_cols, errors='ignore')


Define Targets
SLA breach → classification
Resolution time → regression

In [6]:
target_classification = 'sla_breach_flag'
target_regression = 'resolution_time_hours'

Separate Features and Targets

In [7]:
X = df.drop(columns=[target_classification, target_regression])
y_class = df[target_classification]
y_reg = df[target_regression]


Encode LOW / MEDIUM Cardinality Categoricals

In [8]:
X_encoded = pd.get_dummies(
    X,
    drop_first=True
)


Preserve Time Ordering (Preparation for Split)

In [9]:
df_sorted = df.sort_values('opened_at')


Save Preprocessed Dataset

In [10]:
df_sorted.to_csv(
    "../data/processed/incident_preprocessed.csv",
    index=False
)
